In [ ]:
import sys
import os

# Install libraries specifically for your notebook's kernel
!{sys.executable} -m pip install pandas librosa soundfile numpy tqdm

import pandas as pd
import librosa
import numpy as np
import soundfile as sf
from tqdm import tqdm

print("✅ Phase 1: Libraries are ready!")

In [ ]:
import os
import pandas as pd
import librosa
import numpy as np
import soundfile as sf
from tqdm import tqdm

# 1. PATHS
base_dir = "./tabla_solo_1.0"
wav_dir = os.path.join(base_dir, "wav")
map_dir = os.path.join(base_dir, "onsMap")
output_base = "./data/raw_strokes"

# 2. SYLLABLE MAPPING
SYLLABLE_MAP = {
    'ba': 'Dha', 'da': 'Dha', 'dha': 'Dha',
    'din': 'Dhin', 'dhin': 'Dhin',
    'ga': 'Ghe', 'ge': 'Ghe', 'ghe': 'Ghe',
    'na': 'Na', 'naa': 'Na',
    'ta': 'Ta', 'taa': 'Ta',
    'tun': 'Tun', 'tin': 'Tin', 'ti': 'Ti',
    're': 'Re',
    'ka': 'Ki', 'ke': 'Ki', 'ki': 'Ki',
    'te': 'T', 't': 'T', 'kat': 'Kat'
}

def segment_dataset_fixed():
    target_samples = 72000
    total_count = 0
    os.makedirs(output_base, exist_ok=True)

    # Get map files (checking for both .csv and .txt just in case)
    map_files = [f for f in os.listdir(map_dir) if f.endswith('.csv') or f.endswith('.txt')]
    
    if not map_files:
        print(f"❌ No map files found in {map_dir}!")
        return

    print(f"Found {len(map_files)} map files. Starting extraction...\n")

    for map_file in tqdm(map_files, desc="Extracting Strokes"):
        # FIX: Correctly swap the extension to .wav, regardless of what the map file is called
        wav_name = map_file.rsplit('.', 1)[0] + '.wav'
        wav_path = os.path.join(wav_dir, wav_name)
        
        # Diagnostic check
        if not os.path.exists(wav_path):
            print(f"\n⚠️ Skipping: Could not find matching audio -> {wav_path}")
            continue
            
        map_path = os.path.join(map_dir, map_file)
        
        # Load the timestamps (Failsafe for comma vs. space separated data)
        try:
            df = pd.read_csv(map_path, header=None)
            if df.shape[1] < 2: # If it didn't split properly, try splitting by spaces
                df = pd.read_csv(map_path, sep=r'\s+', header=None)
        except Exception as e:
            print(f"\n⚠️ Error reading map {map_file}: {e}")
            continue

        # Load audio file
        try:
            y, sr = librosa.load(wav_path, sr=44100)
        except Exception as e:
            print(f"\n⚠️ Error loading audio {wav_name}: {e}")
            continue
            
        # Process each hit
        for _, row in df.iterrows():
            try:
                raw_code = str(row[1]).strip().lower()
                category = SYLLABLE_MAP.get(raw_code)
                
                if category:
                    start_sample = int(float(row[0]) * sr)
                    end_sample = start_sample + target_samples
                    
                    stroke = y[start_sample:end_sample]
                    
                    if len(stroke) < target_samples:
                        stroke = np.pad(stroke, (0, target_samples - len(stroke)), 'constant')
                    else:
                        stroke = stroke[:target_samples]
                    
                    class_folder = os.path.join(output_base, category)
                    os.makedirs(class_folder, exist_ok=True)
                    
                    out_filename = f"{category}_{total_count}.wav"
                    sf.write(os.path.join(class_folder, out_filename), stroke, sr)
                    total_count += 1
            except Exception:
                pass # Skip malformed rows silently to keep the loop moving
                
    print(f"\n✅ SUCCESS! Created {total_count} stroke files in {output_base}")

# Run it
segment_dataset_fixed()

# Phase 2: UNDERSTAND (Feature Engineering)
Right now, your AI just sees raw audio waves. We need to convert these waves into a mathematical "fingerprint" that a Convolutional Neural Network (CNN) can process.

According to your specific architecture, we are going to create a Dual Representation for every single stroke. We will transform each .wav file into a (13, 13, 2) tensor (think of it like a tiny, 2-layer image):

Layer 1 (The Texture): 13 MFCCs. This captures the physical "timbre" of the drum (the difference between the skin and the black center).

Layer 2 (The Tone): 13 Chroma features. This captures the pitch and resonance (crucial for distinguishing tuned strokes like Na or Tun).

By taking the first 13 frames of these features, we capture the "attack" of the stroke (the first ~150 milliseconds), which contains the most important acoustic DNA of a drum hit.

In [ ]:
import os
import librosa
import numpy as np
import warnings
from tqdm import tqdm

# Ignore librosa's harmless formatting warnings to keep the output clean
warnings.filterwarnings('ignore') 

input_base = "./data/raw_strokes"
output_dir = "./data/features"
os.makedirs(output_dir, exist_ok=True)

# The 12 specific project classes
CLASSES = ['Dha', 'Dhin', 'Ghe', 'Na', 'Ta', 'Tun', 'Tin', 'Ti', 'Re', 'Ki', 'T', 'Kat']
# Assign a unique number to each class (0 to 11) for the AI
class_to_idx = {cls: i for i, cls in enumerate(CLASSES)}

def extract_dual_tensors():
    X = [] # This will hold the (13, 13, 2) matrices
    y = [] # This will hold the labels (0 to 11)
    
    print("🚀 Starting Phase 2: Extracting (13, 13, 2) Tensors...")
    
    for label in CLASSES:
        class_path = os.path.join(input_base, label)
        if not os.path.exists(class_path):
            continue
            
        files = [f for f in os.listdir(class_path) if f.endswith('.wav')]
        
        for f in tqdm(files, desc=f"Processing {label}"):
            file_path = os.path.join(class_path, f)
            
            try:
                # 1. Load the padded audio
                audio, sr = librosa.load(file_path, sr=44100)
                
                # 2. Extract 13 MFCCs
                mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
                
                # 3. Extract 13 Chroma Features
                chroma = librosa.feature.chroma_stft(y=audio, sr=sr, n_chroma=13)
                
                # 4. Crop to the first 13 frames (The "Attack" of the sound)
                # This neatly gives us a 13x13 grid!
                mfcc_cropped = mfcc[:, :13]
                chroma_cropped = chroma[:, :13]
                
                # Failsafe: Pad if a clip is somehow too short
                if mfcc_cropped.shape[1] < 13:
                    mfcc_cropped = np.pad(mfcc_cropped, ((0,0), (0, 13 - mfcc_cropped.shape[1])))
                    chroma_cropped = np.pad(chroma_cropped, ((0,0), (0, 13 - chroma_cropped.shape[1])))
                
                # 5. Stack them together like a 2-channel image
                dual_tensor = np.stack((mfcc_cropped, chroma_cropped), axis=-1)
                
                X.append(dual_tensor)
                y.append(class_to_idx[label])
                
            except Exception as e:
                # If a single file is corrupted, skip it silently
                continue

    # Convert lists to high-performance NumPy arrays
    X = np.array(X)
    y = np.array(y)
    
    # Save the data for Phase 3
    np.save(os.path.join(output_dir, "X_features.npy"), X)
    np.save(os.path.join(output_dir, "y_labels.npy"), y)
    
    print(f"\n✅ Phase 2 Complete! Feature Tensors Saved.")
    print(f"X shape: {X.shape} (Your Training Data)")
    print(f"y shape: {y.shape} (Your Labels)")

# Run the extraction
extract_dual_tensors()

# Phase 3: TRANSLATE (Training the CNN)
In this phase, we are building the "Brain" of your AI-Powered Rhythmic Transducer.

We are going to construct a Convolutional Neural Network (CNN). Usually, CNNs look at pictures of cats and dogs. But here, your CNN will look at the (13, 13, 2) tensors you just generated.



**Build and Train the Brain**

In [ ]:
import tensorflow as tf
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

# 1. Define the Callbacks (The AI's studying rules)
# Rule 1: If validation accuracy doesn't improve for 3 epochs, cut the learning rate in half.
lr_scheduler = ReduceLROnPlateau(
    monitor='val_accuracy', 
    factor=0.5, 
    patience=3, 
    min_lr=1e-6, 
    verbose=1
)

# Rule 2: If it doesn't improve for 10 epochs, stop early and restore the best weights.
early_stopping = EarlyStopping(
    monitor='val_accuracy', 
    patience=10, 
    restore_best_weights=True,
    verbose=1
)

# 2. Recompile the model to reset the optimizer's state
# We explicitly set the starting learning rate to 0.001
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("🚀 Starting Precision Training with Learning Rate Scheduler...")

# 3. Train the model with the new rules
history_precision = model.fit(
    X_train, y_train, 
    epochs=50, # Increased max epochs since we can slow down now
    batch_size=32, 
    validation_data=(X_test, y_test),
    callbacks=[lr_scheduler, early_stopping]
)

# 4. Save the ultimate 97% version
model.save("./data/tabla_cnn_precision.h5")
print("\n✅ Precision Training Complete! Master Model Saved.")

# Phase 4: RESPOND (The Drum Generator)
This is where the AI-Powered Rhythmic Transducer actually does its job. We are going to take the AI's predictions (Tabla Bols) and map them directly to a standard drum kit using MIDI.

Eventually, we can route this through an LSTM to add "swing" and bar-awareness, but the very first step is building the 1-to-1 Translation Engine.

**Step 1: The Drum Dictionary**
We need to map your 12 Tabla strokes to standard General MIDI (GM) Drum notes.

Bass strokes (Dhin, Ghe) become the Kick Drum.

Sharp rim strokes (Na, Ta) become the Snare.

Muted/fast strokes (Ti, Re, Kat) become Hi-Hats and Rimshots.

**Step 2: Install MIDI Tools**
Create a new cell and run this to install mido, a fantastic Python library for generating MIDI files.

import sys
!{sys.executable} -m pip install mido
print("✅ MIDI tools installed!")

**Step 3: The Translation Script**
This script takes a mock "played sequence" of Tabla strokes, translates them using our new dictionary, and writes an actual .mid file that you can play in any DAW (like FL Studio, Ableton, or Logic).

In [ ]:
import mido
from mido import Message, MidiFile, MidiTrack
import os

# 1. The Translation Dictionary (Tabla Bol -> General MIDI Drum Note)
# 36 = Kick, 38 = Snare, 42 = Closed Hi-Hat, 46 = Open Hi-Hat, etc.
MIDI_MAP = {
    'Dha': [36, 49], # Kick + Crash Cymbal (Heavy beat)
    'Dhin': [36],    # Kick Drum
    'Ghe': [41],     # Low Floor Tom
    'Na': [38],      # Snare Drum
    'Ta': [38],      # Snare Drum
    'Tun': [50],     # High Tom
    'Tin': [51],     # Ride Cymbal
    'Ti': [42],      # Closed Hi-Hat
    'Re': [44],      # Pedal Hi-Hat
    'Ki': [37],      # Side Stick / Rimshot
    'T': [42],       # Closed Hi-Hat
    'Kat': [37]      # Side Stick / Rimshot
}

def translate_to_midi(tabla_sequence, output_filename="./data/generated_drumbeat.mid", tempo_bpm=120):
    # Create a new MIDI file and track
    mid = MidiFile()
    track = MidiTrack()
    mid.tracks.append(track)
    
    # Set the tempo
    microseconds_per_beat = mido.bpm2tempo(tempo_bpm)
    track.append(mido.MetaMessage('set_tempo', tempo=microseconds_per_beat))
    
    # 480 ticks per beat is standard for MIDI resolution
    ticks_per_beat = mid.ticks_per_beat
    
    # We will assume each stroke is an 8th note for this basic test
    ticks_per_stroke = int(ticks_per_beat / 2) 
    
    print(f"🥁 Translating sequence: {tabla_sequence}")
    
    for bol in tabla_sequence:
        notes = MIDI_MAP.get(bol, [42]) # Default to Hi-Hat if unknown
        
        # Turn notes ON (simulating the drum hit)
        for i, note in enumerate(notes):
            # Only the first note in a chord gets the time delay, the rest are 0 (simultaneous)
            time_delay = ticks_per_stroke if i == 0 else 0
            track.append(Message('note_on', note=note, velocity=100, time=time_delay, channel=9)) # Channel 9 is for Drums
            
        # Turn notes OFF
        for i, note in enumerate(notes):
            track.append(Message('note_off', note=note, velocity=64, time=0, channel=9))

    # Save the file
    mid.save(output_filename)
    print(f"✅ Success! Drumbeat saved to: {output_filename}")

# Let's test it with a classic Keherwa Taal (8-beat cycle)
# Sequence: Dha Ge Na Ti | Na Ka Dhin Na
keherwa_taal = ['Dha', 'Ghe', 'Na', 'Ti', 'Na', 'Ki', 'Dhin', 'Na']

translate_to_midi(keherwa_taal)

In [ ]:
import os
import librosa
import numpy as np
import tensorflow as tf
import mido
from mido import Message, MidiFile, MidiTrack
import warnings

warnings.filterwarnings('ignore')

# 1. Configuration & Dictionaries
MODEL_PATH = "./data/tabla_cnn_precision.h5"
CLASSES = ['Dha', 'Dhin', 'Ghe', 'Na', 'Ta', 'Tun', 'Tin', 'Ti', 'Re', 'Ki', 'T', 'Kat']

MIDI_MAP = {
    'Dha': [36, 49], 'Dhin': [36], 'Ghe': [41], 'Na': [38],
    'Ta': [38], 'Tun': [50], 'Tin': [51], 'Ti': [42],
    'Re': [44], 'Ki': [37], 'T': [42], 'Kat': [37]
}

# 2. Load the 94% Accurate Brain
print("🧠 Loading CNN Model...")
model = tf.keras.models.load_model(MODEL_PATH)

def generate_drums_from_audio(audio_file_path, output_midi_path="./data/AI_Drum_Output.mid"):
    print(f"\n🎧 Listening to: {audio_file_path}")
    
    # --- STEP 1: Machine Hearing (Onset Detection) ---
    y, sr = librosa.load(audio_file_path, sr=44100)
    
    # Detect the exact sample index where every stroke begins
    onsets = librosa.onset.onset_detect(y=y, sr=sr, units='samples', backtrack=True)
    print(f"🔪 Detected {len(onsets)} individual strokes in the recording.")
    
    # --- STEP 2: Feature Extraction ---
    tensors = []
    target_samples = 72000
    
    for onset in onsets:
        stroke = y[onset : onset + target_samples]
        
        # Zero-pad if it's the last stroke and too short
        if len(stroke) < target_samples:
            stroke = np.pad(stroke, (0, target_samples - len(stroke)), 'constant')
        else:
            stroke = stroke[:target_samples]
            
        # Extract Dual Representation (13, 13, 2)
        mfcc = librosa.feature.mfcc(y=stroke, sr=sr, n_mfcc=13)[:, :13]
        chroma = librosa.feature.chroma_stft(y=stroke, sr=sr, n_chroma=13)[:, :13]
        
        if mfcc.shape[1] < 13:
            mfcc = np.pad(mfcc, ((0,0), (0, 13 - mfcc.shape[1])))
            chroma = np.pad(chroma, ((0,0), (0, 13 - chroma.shape[1])))
            
        tensor = np.stack((mfcc, chroma), axis=-1)
        tensors.append(tensor)
        
    X_live = np.array(tensors)
    
    # --- STEP 3: CNN Prediction ---
    print("🔮 Translating Tabla Bols...")
    predictions = model.predict(X_live, verbose=0)
    
    predicted_sequence = []
    for pred in predictions:
        class_idx = np.argmax(pred)
        predicted_sequence.append(CLASSES[class_idx])
        
    print(f"📜 AI Transcript: {predicted_sequence}")
    
    # --- STEP 4: MIDI Generation ---
    print("🥁 Writing Drum notation...")
    mid = MidiFile()
    track = MidiTrack()
    mid.tracks.append(track)
    
    # 120 BPM default
    track.append(mido.MetaMessage('set_tempo', tempo=mido.bpm2tempo(120)))
    ticks_per_stroke = int(mid.ticks_per_beat / 2) # 8th notes
    
    for bol in predicted_sequence:
        notes = MIDI_MAP.get(bol, [42]) # Default to Hi-Hat
        
        for i, note in enumerate(notes):
            time_delay = ticks_per_stroke if i == 0 else 0
            track.append(Message('note_on', note=note, velocity=100, time=time_delay, channel=9))
            
        for i, note in enumerate(notes):
            track.append(Message('note_off', note=note, velocity=64, time=0, channel=9))
            
    mid.save(output_midi_path)
    print(f"✅ Success! AI Drum Track saved to: {output_midi_path}")

# --- LET'S TEST IT! ---
# Grab any random .wav file from your dataset to test it on
# Example: replace this with a real file path from your tabla_solo_1.0/wav folder!
test_audio_file = "tabla_solo_1.0\\wav\\pjb_77.wav" # <-- UPDATE THIS PATH

if os.path.exists(test_audio_file):
    generate_drums_from_audio(test_audio_file)
else:
    print(f"\n⚠️ Please update 'test_audio_file' line with a real .wav file path from your dataset to run the final test.")